In [12]:
import os
import requests
import json
from dotenv import load_dotenv
import pandas as pd
from IPython.display import display
import snowflake.connector

load_dotenv()

# conn = snowflake.connector.connect(
#     account=os.getenv("SNOWFLAKE_ACCOUNT"),
#     user=os.getenv("SNOWFLAKE_USER"),
#     password=os.getenv("SNOWFLAKE_PASSWORD"),
#     role=os.getenv("SNOWFLAKE_ROLE"),
#     warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
#     database="RAW",
# )

# cur = conn.cursor()
# cur.execute("""
#     SELECT credits_remaining
#     FROM RAW.PIPELINE.API_USAGE
#     WHERE SOURCE = 'jsearch'
#     ORDER BY RUN_AT DESC
#     LIMIT 1
# """)
# row = cur.fetchone()
# cur.close()

# credits_start = row[0] if row else 200  # fallback to max if no history
# print(f"Starting credits (from Snowflake): {credits_start}")

BASE_URL = "https://jsearch.p.rapidapi.com/search-v2"
HEADERS = {
    "x-rapidapi-key": os.getenv("RAPIDAPI_KEY"),
    "x-rapidapi-host": "jsearch.p.rapidapi.com",
    "Content-Type": "application/json",
}

# credit tracker
credit_log = []
# credits_start = 126  # remaining at start of this session

def fetch(params: dict) -> dict:
    """Make one JSearch request, log credit usage, return data."""
    response = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=30)
    response.raise_for_status()
    
    remaining = int(response.headers.get("x-ratelimit-requests-remaining", 0))
    limit = int(response.headers.get("x-ratelimit-requests-limit", 200))
    used_this_call = credit_log[-1]["remaining"] - remaining if credit_log else credits_start - remaining
    
    credit_log.append({
        "test": params.get("_label", "unlabeled"),
        "query": params.get("query"),
        "remaining": remaining,
        "used_this_call": used_this_call,
        "results": len(response.json().get("data", {}).get("jobs", [])),
    })
    
    print(f"Credits remaining: {remaining} (used {used_this_call} this call) — {len(response.json().get('data', {}).get('jobs', []))} results")
    return response.json().get("data", {})

def show_credit_log():
    display(pd.DataFrame(credit_log))

print("Setup OK — starting with 123 credits")

Setup OK — starting with 123 credits


In [9]:
# Test 1 — bare minimum params, all three queries
# Goal: confirm JSearch returns results with no filters at all
BASE_PARAMS = {
    "num_pages": "1",
    "country": "us",
    "date_posted": "3days",
}
queries = [
    "Data Analyst in New York",
    "Analytics Engineer in New York",
    "Data Engineer in New York",
]

all_jobs = []
for query in queries:
    params = {**BASE_PARAMS, "query": query, "_label": "test1_bare"}
    data = fetch(params)
    jobs = data.get("jobs", [])
    all_jobs.extend(jobs)
    print(f"  {query}: {len(jobs)} results")

print(f"\nTotal: {len(all_jobs)} jobs")
show_credit_log()

Credits remaining: 122 (used 4 this call) — 10 results
  Data Analyst in New York: 10 results
Credits remaining: 121 (used 1 this call) — 10 results
  Analytics Engineer in New York: 10 results
Credits remaining: 120 (used 1 this call) — 10 results
  Data Engineer in New York: 10 results

Total: 30 jobs


,test,query,remaining,used_this_call,results
0,test1_bare,Data Analyst in New York,122,4,10
1,test1_bare,Analytics Engineer in New York,121,1,10
2,test1_bare,Data Engineer in New York,120,1,10


In [7]:
print(json.dumps(jobs[0], indent=2))

{
  "job_id": "hjLvvMqv-oLkGZesAAAAAA==",
  "job_title": "junior Bi Analyst/data analyst/Data Engineer",
  "employer_name": "SynergisticIT",
  "employer_logo": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSZmcVAOMdWiwmokrS0FMOBgGEohAUe8rg1yDiL&s=0",
  "employer_website": "https://www.synergisticit.com",
  "job_publisher": "LinkedIn",
  "job_employment_type": "Full-time",
  "job_employment_types": [
    "FULLTIME"
  ],
  "job_apply_link": "https://www.linkedin.com/jobs/view/junior-bi-analyst-data-analyst-data-engineer-at-synergisticit-4427411114",
  "job_apply_is_direct": false,
  "apply_options": [
    {
      "apply_link": "https://www.linkedin.com/jobs/view/junior-bi-analyst-data-analyst-data-engineer-at-synergisticit-4427411114",
      "is_direct": false,
      "publisher": "LinkedIn"
    }
  ],
  "job_description": "CS/IT/Data Science Graduates or About to be Grads. Get Hired by following a Process!\n\nYou Don\u2019t Need Luck \u2014 You Need Strategy\n\nMany job seekers 

In [13]:
# Cell 2 — field distribution profiler
df = pd.DataFrame(all_jobs)

fields_to_profile = [
    "job_employment_type",
    "job_employment_types",
    "job_publisher",
    "job_country",
    "job_city",
    "job_state",
    "job_is_remote",
]

print(f"Total jobs: {len(df)}\n")
for field in fields_to_profile:
    if field not in df.columns:
        print(f"{field}: NOT IN RESPONSE\n")
        continue
    
    series = df[field]
    null_count = series.isna().sum()
    
    if series.dropna().apply(lambda x: isinstance(x, list)).any():
        exploded = series.explode().dropna()
        val_counts = exploded.value_counts()
    else:
        val_counts = series.value_counts(dropna=False)
    
    print(f"── {field} (nulls: {null_count}/{len(df)}) ──")
    print(val_counts.to_string())
    print()

Total jobs: 30

── job_employment_type (nulls: 0/30) ──
job_employment_type
Full-time                               24
Contractor                               3
Full-time and Part-time                  1
Full-time, Part-time, and Internship     1
Part-time                                1

── job_employment_types (nulls: 0/30) ──
job_employment_types
FULLTIME      26
PARTTIME       3
CONTRACTOR     3
INTERN         1

── job_publisher (nulls: 0/30) ──
job_publisher
LinkedIn               8
ZipRecruiter           4
JobLeads               4
WhatJobs               2
Snagajob               2
EFinancialCareers      1
CUNY Jobs              1
Indeed                 1
Point72 Careers        1
Trigyn Technologies    1
Ladders                1
Learn4Good             1
Trabajo.org            1
Adzuna                 1
Built In NYC           1

── job_country (nulls: 21/30) ──
job_country
NaN    21
US      9

── job_city (nulls: 21/30) ──
job_city
NaN         21
New York     9

── job_state (nul

In [14]:
# Test — full params minus location/radius
EXCLUDE_PUBLISHERS = (
    "Talent.com,Learn4Good,JobLeads,BeBee,WhatJobs,Jobilize,"
    "Jooble,Adzuna,Ladders,Snagajob,Institute Of Data Jobs,"
    "Tech Engineer Jobs,Allied-IT Jobs,United States Jobs Expertini,"
    "Trigyn Technologies,Trigyn,Resume-Library.com,Sign In"
)

for query in queries:
    params = {
        **BASE_PARAMS,
        "query": query,
        "_label": "test_no_location",
        "employment_types": "FULLTIME",
        "job_requirements": "under_3_years_experience,no_experience",
        "exclude_job_publishers": EXCLUDE_PUBLISHERS,
    }
    data = fetch(params)
    jobs = data.get("jobs", [])
    print(f"  {query}: {len(jobs)} results")

print()
show_credit_log()

Credits remaining: 119 (used 7 this call) — 0 results
  Data Analyst in New York: 0 results
Credits remaining: 118 (used 1 this call) — 0 results
  Analytics Engineer in New York: 0 results
Credits remaining: 117 (used 1 this call) — 0 results
  Data Engineer in New York: 0 results



,test,query,remaining,used_this_call,results
0,test_no_location,Data Analyst in New York,119,7,0
1,test_no_location,Analytics Engineer in New York,118,1,0
2,test_no_location,Data Engineer in New York,117,1,0


In [16]:
# Final confirmed params — no job_requirements, no location/radius
no_job_requirements_jobs = []

EXCLUDE_PUBLISHERS = (
    "Talent.com,Learn4Good,JobLeads,BeBee,WhatJobs,Jobilize,"
    "Jooble,Adzuna,Ladders,Snagajob,Institute Of Data Jobs,"
    "Tech Engineer Jobs,Allied-IT Jobs,United States Jobs Expertini,"
    "Trigyn Technologies,Trigyn,Resume-Library.com,Sign In"
)

FINAL_PARAMS = {
    "num_pages": "1",
    "country": "us",
    "date_posted": "3days",
    "employment_types": "FULLTIME",
    "exclude_job_publishers": EXCLUDE_PUBLISHERS,
}

for query in queries:
    params = {**FINAL_PARAMS, "query": query, "_label": "no_job_requirements"}
    data = fetch(params)
    batch = data.get("jobs", [])
    no_job_requirements_jobs.extend(batch)
    print(f"  {query}: {len(batch)} results")

print(f"\nTotal: {len(no_job_requirements_jobs)}")
show_credit_log()

Credits remaining: 113 (used 1 this call) — 9 results
  Data Analyst in New York: 9 results
Credits remaining: 112 (used 1 this call) — 4 results
  Analytics Engineer in New York: 4 results
Credits remaining: 111 (used 1 this call) — 7 results
  Data Engineer in New York: 7 results

Total: 20


,test,query,remaining,used_this_call,results
0,test_no_location,Data Analyst in New York,119,7,0
1,test_no_location,Analytics Engineer in New York,118,1,0
2,test_no_location,Data Engineer in New York,117,1,0
3,test_no_requirements,Data Analyst in New York,116,1,7
4,test_no_requirements,Analytics Engineer in New York,115,1,4
5,test_no_requirements,Data Engineer in New York,114,1,7
6,no_job_requirements,Data Analyst in New York,113,1,9
7,no_job_requirements,Analytics Engineer in New York,112,1,4
8,no_job_requirements,Data Engineer in New York,111,1,7


In [17]:
df_final = pd.DataFrame(no_job_requirements_jobs)

# publisher distribution
print("── Publisher ──")
print(df_final["job_publisher"].value_counts().to_string())

# title list — main thing we care about
print("\n── Job Titles ──")
print(df_final["job_title"].to_string())

# employment type sanity check
print("\n── Employment Type ──")
print(df_final["job_employment_type"].value_counts().to_string())

── Publisher ──
job_publisher
LinkedIn             7
ZipRecruiter         5
Indeed               2
EFinancialCareers    1
CUNY Jobs            1
Point72 Careers      1
Trabajo.org          1
The BIG Jobsite      1
Built In NYC         1

── Job Titles ──
0          junior Bi Analyst/data analyst/Data Engineer
1     Experienced Data Analysis Associate (Hybrid NY...
2          junior Bi Analyst/data analyst/Data Engineer
3     Business Data and Reporting Analyst, Level 1 -...
4         Intern - Financial Data and Reporting Analyst
5                 Data Management - Data Analyst-IT III
6           Data Sourcing & Strategy Operations Analyst
7         Data Analyst III - Regulatory Reporting - FBG
8                              Health Care Data Analyst
9          junior Bi Analyst/data analyst/Data Engineer
10         junior Bi Analyst/data analyst/Data Engineer
11                     Technical Analyst - Python / SQL
12       Senior Energy Engineer: Value-Driven Analytics
13         junior

In [19]:
print(df_final[df_final["job_title"].str.contains("junior Bi", case=False)][["job_id", "job_title", "employer_name"]])

                      job_id                                     job_title  \
0   hjLvvMqv-oLkGZesAAAAAA==  junior Bi Analyst/data analyst/Data Engineer   
2   W2hxRzexR2zspQHnAAAAAA==  junior Bi Analyst/data analyst/Data Engineer   
9   W2hxRzexR2zspQHnAAAAAA==  junior Bi Analyst/data analyst/Data Engineer   
10  hjLvvMqv-oLkGZesAAAAAA==  junior Bi Analyst/data analyst/Data Engineer   
13  hjLvvMqv-oLkGZesAAAAAA==  junior Bi Analyst/data analyst/Data Engineer   
14  W2hxRzexR2zspQHnAAAAAA==  junior Bi Analyst/data analyst/Data Engineer   

    employer_name  
0   SynergisticIT  
2   SynergisticIT  
9   SynergisticIT  
10  SynergisticIT  
13  SynergisticIT  
14  SynergisticIT  
